# pyGeoMapper — Notebook 4: 3D Geological Block Diagram, Cutaway & Fence Slices, and Interactive Viewer

**Site**: Sheep Mountain Anticline, Bighorn Basin, Wyoming, USA (`EPSG:32612`, UTM Zone 12N, `160 × 160` grid at `20 m` resolution)  
**Author**: Larry Syu-Heng Lai  
**License**: AGPL-3.0-or-later (`pyGeoMapper` is an independent Rust + Python geoscience toolkit and is distinct from `The-Magicians-Code/PyGeoMapper`.)

---

### Overview
This notebook demonstrates **`viewer_3d` + `pygeomapper._core` 3D mesh engine** of `pyGeoMapper`: building and visualizing **3D shallow geological block diagrams** with projected subsurface stratigraphy, planar/listric fault offsets, cutaway quadrants, intersecting 3D fence diagrams, binary glTF 2.0 (`.glb`) / `.ply` / `.obj` 3D mesh export, `forge3d` terrain draping, and a **100% offline interactive HTML5/WebGL 3D block viewer** (`outputs/sheep_mtn_3d_block_viewer.html`).

1. **Load Sheep Mountain Anticline Dataset & Auto-Mapped Units**:
   - USGS 3DEP DEM (`1134.0 m` to `1512.8 m` NAVD88, superposed Bighorn River canyon).
   - USGS SGMC Reference Formations (`0: Mm` Madison Limestone through `6: Qa` Quaternary Alluvium) and Supervised Hybrid Auto-Mapped Formations.
2. **Rust 3D Block Volume & Watertight Mesh Engine (`pygeomapper._core`)**:
   - Calibrates the 2D structural fold elevation surface $h_{\text{struct}}(y, x)$ from surface contacts, 3-point bed attitudes, and the published SW-dipping ~50° basement thrust (`strike ≈ 140°` RHR → dip toward 230° SW; Bellahsen; Stanton and Erslev 2004), reconstructed once via `cross_section.reconstruct_planar_fault` and shared with notebook 3.
   - Evaluates the 3D voxel stratigraphic volume `(160 × 160 × 64)` down to `750 m` elevation (`~384 m` below the Bighorn River canyon floor) in parallel Rust (`build_3d_block_volume_rs`).
   - Generates complete 3D triangle meshes (`build_block_diagram_meshes_rs`) for the top topographic surface, 4 vertical cross-section side walls (South, North, West, East), base plate, internal cutaway walls/bench, and 3D thrust fault plane.
3. **3D Cutaway Block & Intersecting Fence Diagrams (`slice_3d_fence_diagram`)**:
   - Excavates the SW quadrant to expose the folded Paleozoic–Mesozoic core and thrust offset inside the 3D block.
   - Marks A–A' and B–B' (~60° apart) on the opaque volumetric block and shows those same two faulted 3D sections together — not a three-fence board.
4. **Multi-Modal Surface Drape Comparison & Interactive HTML5/WebGL 3D Viewer**:
   - Compares draping USGS Reference Geology, Auto-Mapped Geology, Sentinel-2 False-Color (`SWIR1-NIR-Red`), and 2D Mexican-Hat CWT Multi-Scale Complexity onto the 3D block diagram.
   - Exports binary `.glb`, `.ply`, and `.obj` 3D meshes and launches the self-contained interactive 3D HTML viewer (`outputs/sheep_mtn_3d_block_viewer.html`).

### About this notebook
- **Purpose:** 3D geological block diagram, cutaway and fence diagrams, multi-drape comparison and offline interactive viewer for Sheep Mountain Anticline, WY.
- **Data used:** `example_data/sheep_mtn_dem_3dep.tif`, `sheep_mtn_geol_units.tif`, `sheep_mtn_s2.tif`, `sheep_mtn_alphaearth_2024.tif`, `sheep_mtn_units.json`.
- **Figures written:** `image/3d_block_diagram.png` (README: opaque block with A–A′ & B–B′ marked + matching faulted sections), `image/3d_cutaway_and_fence.png` (cutaway + single A–A′ fence), `image/surface_drape_comparison.png`. Meshes and the HTML viewer go to `outputs/` (not tracked by git).
- **How to re-run:** run all cells from `notebooks/` or the repo root. Figures are saved to `image/` with `plotting.save_figure` and are not embedded in the notebook. `forge3d` comes with `pip install pygeomapper`; conda users without it can `pip install forge3d`, otherwise the notebook uses the built-in fallback renderer.

In [1]:
import json
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np

import pygeomapper as pgm
from pygeomapper import (
    auto_mapping,
    cross_section,
    plotting,
    raster_io,
    terrain_analysis,
    viewer_3d,
)

ROOT = Path.cwd() if (Path.cwd() / "example_data").exists() else Path.cwd().parent   # repo root (run from notebooks/ or the root)

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 170

DATA_DIR = ROOT / "example_data"
OUT_DIR = ROOT / "outputs"
IMG_DIR = ROOT / "image"
for d in (OUT_DIR, IMG_DIR):
    d.mkdir(parents=True, exist_ok=True)

dem = raster_io.load_geotiff(DATA_DIR / "sheep_mtn_dem_3dep.tif", name="sheep_dem", units="m")
geol_ref = raster_io.load_geotiff(DATA_DIR / "sheep_mtn_geol_units.tif", name="sheep_geol", units="unit_id")
s2_stack = raster_io.load_multiband_geotiff(
    DATA_DIR / "sheep_mtn_s2.tif",
    band_names=["blue", "green", "red", "nir", "swir1", "swir2"],
    name="sheep_s2",
    units="reflectance",
)
with open(DATA_DIR / "sheep_mtn_units.json", encoding="utf-8") as f:
    units_meta = json.load(f)
unit_names = {u["unit_id"]: f"{u['code']}: {u['name']}" for u in units_meta["units"]}

print(f"pyGeoMapper version: {pgm.__version__} (forge3d available: {viewer_3d.HAS_FORGE3D})")
print(f"DEM grid: {dem.shape} at {dem.dx:.0f} m spacing, elevation {np.nanmin(dem.data):.1f} - {np.nanmax(dem.data):.1f} m NAVD88")
print(f"Formations loaded: {len(units_meta['units'])}")

pyGeoMapper version: 0.2.0 (forge3d available: True)
DEM grid: (160, 160) at 20 m spacing, elevation 1134.0 - 1512.8 m NAVD88
Formations loaded: 7


## 1. Reconstruct Toggleable Auto-Mapped Units (`feature_mode='engineered'` vs. `feature_mode='alphaearth'`)

### 1. What This Step Does & Why We Use This Method
- We load both the **29-band Engineered Physics + Spectral + 2D CWT stack** (`feature_mode='engineered'`) and the **64-band AlphaEarth Foundations Unit-Sphere Embedding stack** (`feature_mode='alphaearth'`, `sheep_mtn_alphaearth_2024.tif`) and train the hybrid supervised classifier on the spatial-block split.
- Both auto-mapped geological maps (`geol_auto_eng` and `geol_auto_ae`) are then built into 3D block models and added as **toggleable surface drapes** inside the interactive 3D HTML5/WebGL viewer (`outputs/sheep_mtn_3d_block_viewer.html`).


In [2]:
ae_stack = raster_io.load_multiband_geotiff(
    DATA_DIR / "sheep_mtn_alphaearth_2024.tif",
    dequantize_alphaearth=True,
    name="sheep_alphaearth",
)
feature_stack = auto_mapping.build_mapping_feature_stack(
    dem,
    spectral_stack=s2_stack,
    cwt_scales_m=[30.0, 60.0, 120.0, 200.0],
    feature_mode="engineered",
)
split = auto_mapping.create_spatial_block_split(
    dem,
    mode="west_east",
    test_fraction=0.35,
    buffer_px=2,
)
sup_eng = auto_mapping.train_predict_supervised_units(
    feature_stack,
    geol_ref,
    train_mask=split["train_mask"],
    method="hybrid",
    smooth_radius_px=2,
    feature_mode="engineered",
)
sup_ae = auto_mapping.train_predict_supervised_units(
    feature_stack,
    geol_ref,
    train_mask=split["train_mask"],
    method="hybrid",
    smooth_radius_px=2,
    embedding_stack=ae_stack,
    feature_mode="alphaearth",
)
geol_auto = sup_eng["pred_labels"]
geol_auto_ae = sup_ae["pred_labels"]

eval_eng_full = auto_mapping.evaluate_mapping_accuracy(geol_ref, geol_auto, unit_names=unit_names)
eval_ae_full = auto_mapping.evaluate_mapping_accuracy(geol_ref, geol_auto_ae, unit_names=unit_names)
eval_full = eval_ae_full
attitudes_3pt = cross_section.estimate_contact_attitudes(geol_ref, dem, window_m=320.0)
print(
    f"Engineered (29B) Full-Scene Agreement: {eval_eng_full['overall_accuracy']:.3f} | "
    f"AlphaEarth (64B) Full-Scene Agreement: {eval_ae_full['overall_accuracy']:.3f} | "
    f"3-Point Contact Attitudes: {len(attitudes_3pt)}"
)


Engineered (29B) Full-Scene Agreement: 0.883 | AlphaEarth (64B) Full-Scene Agreement: 0.955 | 3-Point Contact Attitudes: 41


## 2. Build the 3D Shallow Geological Block Model in Rust (`build_3d_block_diagram`)

### 1. What This Step Does & Why We Use This Method
- **The Goal:** Convert a 2D surface geological map, 3DEP Digital Elevation Model, and 3-point strike/dip attitudes into a watertight **3D volumetric block diagram** (`160 × 160 × 64` voxels down to `750 m` elevation) that reveals how folded beds and thrust faults project underground.

### 2. Governing Equations & Physical Meaning
- **3D Stratigraphic Potential Field (`calibrate_3d_structural_horizon` & `build_3d_block_volume_rs`):**
  $$\zeta(x, y, z) = z - h_{\text{struct}}(x, y) - \Delta z_{\text{fault}}(x, y, z)$$
  - $z$: Voxel elevation above sea level (meters NAVD88).
  - $h_{\text{struct}}(x, y)$: Smooth 2D structural fold reference surface calibrated from surface formation contacts and 3-point bedding attitudes.
  - $\Delta z_{\text{fault}}(x, y, z)$: Hanging-wall vertical throw ($+75\text{ m}$) above the dipping reverse/thrust fault plane $z_{\text{fault}}(x, y) = z_0 - d_{\text{dip}}\tan\delta_{\text{fault}}$.
  - Each voxel $(x, y, z)$ is assigned to formation $k$ according to its stratigraphic height $\zeta(x, y, z) \in [\zeta_{k-1}, \zeta_k)$.


In [3]:
# Same reconstructed basement thrust as notebook 3 (published attitude; inferred tip/throw).
start_xy = (726900.0, 4942800.0)
end_xy = (729700.0, 4944600.0)
tip = cross_section.locate_fault_tip_along_section(dem, start_xy, end_xy)
throw_inferred_m = float(np.clip(0.5 * tip["section_relief_m"], 80.0, 200.0))
fault_spec = cross_section.reconstruct_planar_fault(
    dem,
    strike_deg=140.0,
    dip_deg=50.0,
    throw_m=throw_inferred_m,
    name="Sheep Mountain basement thrust",
    tip_section=(start_xy, end_xy, tip["distance_m"]),
    z0=tip["z"],
    published={
        "strike_deg": 140.0,
        "dip_deg": 50.0,
        "dip_direction": "SW",
        "source": "Bellahsen; Stanton and Erslev 2004",
    },
    inferred={
        "tip_distance_m": tip["distance_m"],
        "tip_xy": (tip["x"], tip["y"]),
        "z0_m": tip["z"],
        "throw_m": throw_inferred_m,
        "note": "Tip = DEM steep-forelimb toe on A–A'; throw ≈ ½ crest–basin relief for shallow demo",
    },
)
print("Fault published:", fault_spec["published"])
print("Fault inferred:", fault_spec["inferred"])

# Faulted block for README (include_fault=False remains available as a user toggle).
block_ref = viewer_3d.build_3d_block_diagram(
    dem=dem,
    geol_labels=geol_ref,
    units_metadata=units_meta,
    attitudes=attitudes_3pt,
    z_bottom=750.0,
    nz=64,
    fault_spec=fault_spec,
    include_fault=True,
    cutaway_fraction=(0.48, 0.48, 0.36),
)

block_auto_eng = viewer_3d.build_3d_block_diagram(
    dem=dem,
    geol_labels=geol_auto,
    units_metadata=units_meta,
    attitudes=attitudes_3pt,
    z_bottom=750.0,
    nz=64,
    fault_spec=fault_spec,
    include_fault=True,
    cutaway_fraction=(0.48, 0.48, 0.36),
)

block_auto = viewer_3d.build_3d_block_diagram(
    dem=dem,
    geol_labels=geol_auto_ae,
    units_metadata=units_meta,
    attitudes=attitudes_3pt,
    z_bottom=750.0,
    nz=64,
    fault_spec=fault_spec,
    include_fault=True,
    cutaway_fraction=(0.48, 0.48, 0.36),
)

print(f"Full 3D Block Mesh: {block_ref['mesh']['n_vertices']:,} vertices, {block_ref['mesh']['n_triangles']:,} triangles")
print(f"Cutaway 3D Block Mesh: {block_ref['cutaway_mesh']['n_vertices']:,} vertices, {block_ref['cutaway_mesh']['n_triangles']:,} triangles")
print("\nSubsurface 3D Voxel Volume Breakdown (Sheep Mountain Block down to 750 m):")
for st in block_ref["unit_volume_stats"]:
    print(
        f"  [{st['id']}] {st['code']:4s} ({st['name']:30s}): "
        f"{st['voxel_count']:7,d} voxels ({st['volume_fraction']*100:5.1f}%, {st['volume_km3']:.3f} km^3)"
    )


Fault published: {'strike_deg': 140.0, 'dip_deg': 50.0, 'dip_direction': 'SW', 'source': 'Bellahsen; Stanton and Erslev 2004'}
Fault inferred: {'tip_distance_m': 2339.4220650532666, 'tip_xy': (728867.8714859438, 4944065.0602409635), 'z0_m': 1185.0470345340436, 'throw_m': 139.63056564679664, 'note': "Tip = DEM steep-forelimb toe on A–A'; throw ≈ ½ crest–basin relief for shallow demo"}
Full 3D Block Mesh: 128,560 vertices, 105,996 triangles
Cutaway 3D Block Mesh: 153,475 vertices, 109,560 triangles

Subsurface 3D Voxel Volume Breakdown (Sheep Mountain Block down to 750 m):
  [0] Mm   (Madison Limestone             ): 677,607 voxels ( 63.8%, 3.333 km^3)
  [1] IPta (Tensleep Sandstone & Amsden Fm): 135,427 voxels ( 12.8%, 0.666 km^3)
  [2] TRPc (Chugwater & Goose Egg Fms     ): 103,789 voxels (  9.8%, 0.511 km^3)
  [3] Jsg  (Sundance & Gypsum Spring Fms  ):  69,389 voxels (  6.5%, 0.341 km^3)
  [4] KJcm (Cloverly & Morrison Fms       ):  49,934 voxels (  4.7%, 0.246 km^3)
  [5] Kmt  (Mowry

## 3. README figure: opaque 3D block + matching A–A′ & B–B′ sections (`image/3d_block_diagram.png`)

### How to Read the Visual Output
- **Left (a):** Opaque solid volumetric 3D geologic block with **both** $A\text{–}A'$ and $B\text{–}B'$ (~60° different) marked on the top surface. Non-fault unit contacts are solid black lines; the basement thrust is a **red** surface trace with dip-direction ticks.
- **Right (b):** Matching 3D fences for the same two lines, with the reconstructed fault as a **transparent red surface** dipping underground.
- **Shared camera:** Both panels use the same view (~30° clockwise from a SE face-on angle) so more of the south–north block side is visible. No unfaulted twin; no extra unused fences.


In [4]:
# Same two sections as notebook 3 / image/cross_section.png (~60° apart).
# Camera = original "good" SE view (−58°) + 20° clockwise → −78° / elev 28°.
# Layout: (a) opaque block | (b) ONE box with both A–A′ & B–B′ fences (not stacked subplots).
start_bb = (728390.0, 4941805.0)
end_bb = (728250.0, 4944795.0)
sections = [
    (start_xy, end_xy, "A", "A'"),
    (start_bb, end_bb, "B", "B'"),
]
fig, _axes = viewer_3d.plot_3d_block_and_section(
    block_ref,
    start_xy,
    end_xy,
    sections=sections,
    vertical_exaggeration=1.7,
    azimuth_deg=-78.0,
    elevation_deg=28.0,
    n_samples=160,
    show_fault_plane=True,
    top_stride=2,
    wall_stride=2,
    section_alpha=1.0,
    fault_alpha=0.10,
    titles=(
        "(a) 3D geologic block with A–A′ & B–B′ section locations",
        f"(b) Matching 3D sections A–A′ & B–B′ + SW-dipping ~{fault_spec['dip_deg']:.0f}° basement thrust (throw +{fault_spec['throw_m']:.0f} m inferred)",
    ),
)
fig_path1 = IMG_DIR / "3d_block_diagram.png"
plotting.save_figure(fig, fig_path1, max_width_px=1500, palette=False)
print(f"Saved {fig_path1}")


Saved image/3d_block_diagram.png


## 4. 3D Cutaway Block & A–A′ / B–B′ Fences (`slice_3d_fence_diagram`)

Internal view of the fold and the reconstructed SW-dipping basement thrust:
1. **3D cutaway block** (`plot_3d_cutaway_block`) with the SW quadrant excavated (fault plane on).
2. **A–A′ and B–B′** fence ribbons (same transects as the README 3D/2D sections) with the fault plane — not a three-fence board.


In [5]:
start_bb = (728390.0, 4941805.0)
end_bb = (728250.0, 4944795.0)
sections = [
    (start_xy, end_xy, "A", "A'"),
    (start_bb, end_bb, "B", "B'"),
]
fence_panels = viewer_3d.slice_3d_fence_diagram(
    block_ref, transects=[(s[0], s[1]) for s in sections], n_samples=160
)
for fp, sec in zip(fence_panels, sections):
    fp["name"] = f"Fence {sec[2]}-{sec[3]}"
    print(
        f"{fp['name']}: length={fp['distance_m'][-1]:.0f} m, "
        f"ribbon triangles={fp['mesh']['n_triangles']:,}"
    )

fig = plt.figure(figsize=(16, 7.2), layout="constrained")
ax1 = fig.add_subplot(121, projection="3d")
viewer_3d.plot_3d_cutaway_block(
    block_ref,
    vertical_exaggeration=1.7,
    azimuth_deg=-78.0,
    elevation_deg=28.0,
    face_stride=1,
    show_fault_plane=True,
    show_legend=True,
    ax=ax1,
    title="(a) 3D cutaway block (fault plane on)",
)

ax2 = fig.add_subplot(122, projection="3d")
viewer_3d.plot_3d_fence_diagram(
    block_ref,
    fence_panels=fence_panels,
    vertical_exaggeration=1.7,
    azimuth_deg=-78.0,
    elevation_deg=28.0,
    show_fault_plane=True,
    show_wireframe_surface=True,
    show_legend=False,
    ax=ax2,
    title="(b) 3D A–A′ & B–B′ fences + reconstructed fault plane",
)
fig_path2 = IMG_DIR / "3d_cutaway_and_fence.png"
plotting.save_figure(fig, fig_path2, max_width_px=1500)
print(f"Saved {fig_path2}")


Fence A-A': length=3329 m, ribbon triangles=13,086


Saved image/3d_cutaway_and_fence.png


## 5. Multi-Modal Surface Drape Comparison & `forge3d` Terrain Adapter

A major advantage of separating the 3D subsurface block mesh from the surface texture is that any co-registered remote-sensing or geomorphic layer can be draped over the 3D topography while preserving the projected subsurface geology on the 4 vertical side walls:
1. **USGS Reference Geological Map** (`block_ref['surface_drape_rgba']`)
2. **Supervised Auto-Mapped Formations** (`block_auto['surface_drape_rgba']`)
3. **Sentinel-2 False-Color Lithologic Composite (`SWIR1 - NIR - Red`)**
4. **2D Mexican-Hat CWT Multi-Scale Topographic Complexity (`pyTopoComplexity`)**

In [6]:
def stretch_band(arr: np.ndarray) -> np.ndarray:
    lo, hi = np.nanpercentile(arr, [2.0, 98.0])
    return np.clip((arr - lo) / max(hi - lo, 1e-9), 0.0, 1.0)

hs = block_ref["hillshade"]
s2_false_rgb = np.stack(
    [
        stretch_band(s2_stack.band("swir1").data),
        stretch_band(s2_stack.band("nir").data),
        stretch_band(s2_stack.band("red").data),
    ],
    axis=-1,
)
# Blend Sentinel-2 false color with multi-azimuth hillshade
s2_drape_f = np.clip(s2_false_rgb * (0.48 + 0.72 * hs.data[:, :, np.newaxis]), 0.0, 1.0)
s2_drape_rgba = np.dstack(
    [(s2_drape_f * 255).astype(np.uint8), np.full(dem.shape, 255, dtype=np.uint8)]
)

# Drape 2D CWT Multi-Scale Complexity on hillshade via forge3d adapter
cwt_mean_grid = feature_stack.band("mean_cwt_complexity")
cwt_drape_f = viewer_3d.drape_overlay_on_hillshade(
    hs, cwt_mean_grid, cmap="magma", alpha=0.75
)
cwt_drape_rgba = np.dstack(
    [(cwt_drape_f * 255).astype(np.uint8), np.full(dem.shape, 255, dtype=np.uint8)]
)
forge_info = viewer_3d.render_with_forge3d(dem, overlay_rgb=cwt_drape_f)
print(f"3D Terrain Drape Backend: {forge_info['backend']} (HAS_FORGE3D={forge_info['has_forge3d']})")

fig = plt.figure(figsize=(15.5, 12.0))
drape_configs = [
    ("(a) USGS SGMC Reference Geology Drape", block_ref["surface_drape_rgba"]),
    ("(b) Auto-Mapped Formations Drape", block_auto["surface_drape_rgba"]),
    ("(c) Sentinel-2 Lithologic False-Color (SWIR1-NIR-Red) Drape", s2_drape_rgba),
    ("(d) 2D Mexican-Hat CWT Multi-Scale Complexity Drape", cwt_drape_rgba),
]
for idx, (sub_title, d_rgba) in enumerate(drape_configs, start=1):
    ax = fig.add_subplot(2, 2, idx, projection="3d")
    viewer_3d.plot_3d_block_diagram(
        block_ref,
        surface_drape_rgba=d_rgba,
        vertical_exaggeration=1.65,
        azimuth_deg=-55.0,
        elevation_deg=29.0,
        top_stride=2,
        wall_stride=1,
        show_attitudes=(idx <= 2),
        show_legend=(idx == 1),
        ax=ax,
        title=sub_title,
    )
plt.tight_layout()
fig_path3 = IMG_DIR / "surface_drape_comparison.png"
plotting.save_figure(fig, fig_path3, max_width_px=1200)
print(f"Saved {fig_path3}")

3D Terrain Drape Backend: forge3d (HAS_FORGE3D=True)


Saved image/surface_drape_comparison.png


## 6. Export 3D Meshes (`.glb`, `.ply`, `.obj`) & Launch Offline Interactive 3D Block Viewer (`outputs/sheep_mtn_3d_block_viewer.html`)

### Toggleable Surface Drapes in the Interactive 3D Viewer
Both mapping methods remain **first-class toggleable options** inside the interactive HTML5/WebGL viewer dropdown:
1. `Geological Map (Primary)` — USGS SGMC Reference Map
2. `Auto-Mapped Units (Engineered 29-Band)` — Physics + Spectral + 2D CWT Auto-Map (`feature_mode='engineered'`)
3. `Auto-Mapped Units (AlphaEarth 64-Band)` — AlphaEarth Unit-Sphere Embedding Auto-Map (`feature_mode='alphaearth'`)
4. `Sentinel-2 False-Color (SWIR1-NIR-Red)` — Optical lithologic composite
5. `2D CWT Multi-Scale Complexity` — Multi-scale wavelet energy drape


In [7]:
glb_res = viewer_3d.export_3d_block_mesh(
    block_ref, OUT_DIR / "sheep_mtn_block.glb", use_cutaway=False, vertical_exaggeration=1.5
)
ply_res = viewer_3d.export_3d_block_mesh(
    block_ref, OUT_DIR / "sheep_mtn_cutaway.ply", use_cutaway=True, vertical_exaggeration=1.5
)
obj_res = viewer_3d.export_3d_block_mesh(
    block_ref, OUT_DIR / "sheep_mtn_block.obj", use_cutaway=False, vertical_exaggeration=1.5
)
print("Exported 3D Meshes:")
for r in (glb_res, ply_res, obj_res):
    print(
        f"  - {r['path']} ({r['format'].upper()}): {r['n_vertices']:,} vertices, "
        f"{r['n_triangles']:,} triangles, {r['file_size_bytes']/1024:.1f} KB"
    )

extra_drapes = {
    "Auto-Mapped Units (Engineered 29-Band)": block_auto_eng["surface_drape_rgba"],
    "Auto-Mapped Units (AlphaEarth 64-Band)": block_auto["surface_drape_rgba"],
    "Sentinel-2 False-Color (SWIR1-NIR-Red)": s2_drape_rgba,
    "2D CWT Multi-Scale Complexity": cwt_drape_rgba,
}
html_meta = viewer_3d.export_interactive_block_html(
    block_ref,
    output_html=OUT_DIR / "sheep_mtn_3d_block_viewer.html",
    extra_surface_drapes=extra_drapes,
    grid_stride=2,
    title="Sheep Mountain Anticline, WY — 3D Geological Block & Subsurface Viewer",
)
print(
    f"\nInteractive 3D HTML Viewer generated: {html_meta['html_path']} "
    f"({html_meta['file_size_bytes']/1024:.1f} KB, drapes={html_meta['surface_drapes']})"
)
viewer_3d.launch_block_viewer(
    block_ref,
    output_html=OUT_DIR / "sheep_mtn_3d_block_viewer.html",
    extra_surface_drapes=extra_drapes,
    grid_stride=2,
)


Exported 3D Meshes:
  - outputs/sheep_mtn_block.glb (GLB): 128,560 vertices, 105,996 triangles, 4758.4 KB
  - outputs/sheep_mtn_cutaway.ply (PLY): 153,475 vertices, 109,560 triangles, 11510.6 KB
  - outputs/sheep_mtn_block.obj (OBJ): 128,560 vertices, 105,996 triangles, 13467.7 KB

Interactive 3D HTML Viewer generated: outputs/sheep_mtn_3d_block_viewer.html (1015.3 KB, drapes=['Geological Map (Primary)', 'Auto-Mapped Units (Engineered 29-Band)', 'Auto-Mapped Units (AlphaEarth 64-Band)', 'Sentinel-2 False-Color (SWIR1-NIR-Red)', '2D CWT Multi-Scale Complexity'])
